In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import matplotlib.cm as cm
import plotly.express as px
from sklearn.decomposition import PCA

In [2]:
def sample_cohort(group, n=10000):
    if len(group) > n:
        return group.sample(n=n, random_state=42)
    else:
        return group

# Combined 3DINO vs MedSAM2

In [ ]:
# ---------------------------
# Compute t-SNE embeddings once per model (reused across all four color-by variables)
# ---------------------------
def load_merged(csv_path):
    df = pd.read_csv(csv_path)
    df_params = pd.read_csv("../../local/input/all_img_params.csv")
    return pd.merge(df, df_params, left_on='id', right_on='id', how='inner')

# 3DINO
df_dino = load_merged("../../local/output/combined_data_dino.csv")
dino_feature_cols = [f"Feature{i}" for i in range(1, 1025)]
df_dino_sampled = df_dino.groupby(cohort_col, group_keys=False).apply(sample_cohort)
X_dino = df_dino_sampled[dino_feature_cols].values
X_dino_pca = PCA(n_components=256, random_state=42).fit_transform(X_dino)
X_dino_tsne = TSNE(n_components=2, random_state=42, init='random', perplexity=7).fit_transform(X_dino_pca)
df_dino_sampled['tsne-1'] = X_dino_tsne[:, 0]
df_dino_sampled['tsne-2'] = X_dino_tsne[:, 1]

# MedSAM2
df_medsam2 = load_merged("../../local/output/combined_data_medsam2.csv")
medsam2_feature_cols = [f"Feature{i}" for i in range(1, 257)]
df_medsam2_sampled = df_medsam2.groupby(cohort_col, group_keys=False).apply(sample_cohort)
X_medsam2 = df_medsam2_sampled[medsam2_feature_cols].values
X_medsam2_tsne = TSNE(n_components=2, random_state=42, perplexity=7).fit_transform(X_medsam2)
df_medsam2_sampled['tsne-1'] = X_medsam2_tsne[:, 0]
df_medsam2_sampled['tsne-2'] = X_medsam2_tsne[:, 1]

In [ ]:
# ---------------------------
# Combined figure: rows = color-by variable, cols = model.
# Color is the ONLY encoding channel (no fill/no-fill split by diagnosis) -
# each point is a plain filled circle so color unambiguously means one thing per row.
# ---------------------------
CAT_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEPARATOR_COLOR = "#c3c2b7"

def sort_key(v):
    try:
        return (0, float(v))
    except (TypeError, ValueError):
        return (1, str(v))

row_specs = [
    (label_col, "Diagnosis", {0: "CN", 1: "AD"}),
    (cohort_col, "Cohort", None),
    (scanner_col, "Scanner Manufacturer", None),
    (mfs_col, "Field Strength (T)", None),
]
model_specs = [
    ("3DINO", df_dino_sampled),
    ("MedSAM2", df_medsam2_sampled),
]

fig, axes = plt.subplots(
    len(row_specs), len(model_specs),
    figsize=(6 * len(model_specs), 4.5 * len(row_specs)),
    squeeze=False,
)

for i, (col_key, row_title, value_map) in enumerate(row_specs):
    values = sorted(
        set(df_dino_sampled[col_key].unique()) | set(df_medsam2_sampled[col_key].unique()),
        key=sort_key,
    )
    color_map = {v: CAT_COLORS[k % len(CAT_COLORS)] for k, v in enumerate(values)}
    display_name = (lambda v: value_map.get(v, v)) if value_map else (lambda v: v)

    legend_handles = {}
    for j, (model_name, df_sampled) in enumerate(model_specs):
        ax = axes[i, j]
        # Largest category first (bottom layer) so minority categories plot last
        # and stay visible on top instead of being buried under overlap.
        counts = df_sampled[col_key].value_counts()
        plot_order = sorted(values, key=lambda v: counts.get(v, 0), reverse=True)
        for v in plot_order:
            subset = df_sampled[df_sampled[col_key] == v]
            if subset.empty:
                continue
            sc = ax.scatter(
                subset['tsne-1'], subset['tsne-2'],
                color=color_map[v],
                s=14,
                alpha=0.7,
                edgecolors='none',
                label=str(display_name(v)),
            )
            legend_handles[str(display_name(v))] = sc
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if i == 0:
            ax.set_title(model_name, fontsize=18, pad=10)
        if j == 0:
            ax.set_ylabel(row_title, fontsize=15, labelpad=10)

    # Legend order follows the fixed `values` order (not the per-panel plot order)
    # so it stays consistent between the two model columns.
    ordered_labels = [str(display_name(v)) for v in values if str(display_name(v)) in legend_handles]
    ordered_handles = [legend_handles[l] for l in ordered_labels]
    axes[i, -1].legend(
        ordered_handles, ordered_labels,
        loc='center left',
        bbox_to_anchor=(1.03, 0.5),
        fontsize=11,
        frameon=False,
        markerscale=1.8,
        title=row_title,
        title_fontsize=12,
    )

fig.subplots_adjust(hspace=0.2, wspace=0.15, right=0.82)

# ---------------------------
# Separator lines between rows/cols (grid panels touch, so spines alone don't read as separate cells)
# ---------------------------
fig.canvas.draw()  # finalize axes positions before reading them

left_edge = min(axes[i, 0].get_position().x0 for i in range(len(row_specs)))
right_edge = max(axes[i, -1].get_position().x1 for i in range(len(row_specs)))
top_edge = max(axes[0, j].get_position().y1 for j in range(len(model_specs)))
bottom_edge = min(axes[-1, j].get_position().y0 for j in range(len(model_specs)))

# vertical separators between model columns
for j in range(len(model_specs) - 1):
    x_sep = (axes[0, j].get_position().x1 + axes[0, j + 1].get_position().x0) / 2
    fig.add_artist(plt.Line2D(
        [x_sep, x_sep], [bottom_edge, top_edge],
        transform=fig.transFigure, color=SEPARATOR_COLOR, linewidth=1,
    ))

# horizontal separators between row groups
for i in range(len(row_specs) - 1):
    y_sep = (axes[i, 0].get_position().y0 + axes[i + 1, 0].get_position().y1) / 2
    fig.add_artist(plt.Line2D(
        [left_edge, right_edge], [y_sep, y_sep],
        transform=fig.transFigure, color=SEPARATOR_COLOR, linewidth=1,
    ))

fig.savefig("../../local/figures/tsne_3dino_medsam2_combined.png", dpi=300, bbox_inches='tight')
plt.show()

# BrainIAC, SAM-Med3D and Medcial-SAM2

In [ ]:
# ---------------------------
# Compute t-SNE embeddings once per model (reused across all four color-by variables)
# ---------------------------
def load_merged(csv_path):
    df = pd.read_csv(csv_path)
    df_params = pd.read_csv("../../local/input/all_img_params.csv")
    return pd.merge(df, df_params, left_on='id', right_on='id', how='inner')

# SAM-Med3D
df_sammed = load_merged("../../local/output/combined_data_sammed.csv")
sammed_feature_cols = [f"Feature{i}" for i in range(1, 385)]
df_sammed_sampled = df_sammed.groupby(cohort_col, group_keys=False).apply(sample_cohort)
X_sammed = df_sammed_sampled[sammed_feature_cols].values
X_sammed_tsne = TSNE(n_components=2, random_state=42, init='random', perplexity=15).fit_transform(X_sammed)
df_sammed_sampled['tsne-1'] = X_sammed_tsne[:, 0]
df_sammed_sampled['tsne-2'] = X_sammed_tsne[:, 1]

# BrainIAC
df_brainiac = load_merged("../../local/output/combined_data_brainiac.csv")
brainiac_feature_cols = [f"Feature{i}" for i in range(1, 769)]
df_brainiac_sampled = df_brainiac.groupby(cohort_col, group_keys=False).apply(sample_cohort)
X_brainiac = df_brainiac_sampled[brainiac_feature_cols].values
X_brainiac_tsne = TSNE(n_components=2, random_state=42, perplexity=50).fit_transform(X_brainiac)
df_brainiac_sampled['tsne-1'] = X_brainiac_tsne[:, 0]
df_brainiac_sampled['tsne-2'] = X_brainiac_tsne[:, 1]


# MedicalSAM2
df_medsam2 = load_merged("../../local/output/combined_data_medicalsam2.csv")
medsam2_feature_cols = [f"Feature{i}" for i in range(1, 257)]
df_medsam2_sampled = df_medsam2.groupby(cohort_col, group_keys=False).apply(sample_cohort)
X_medsam2 = df_medsam2_sampled[medsam2_feature_cols].values
X_medsam2_tsne = TSNE(n_components=2, random_state=42, perplexity=15).fit_transform(X_medsam2)
df_medsam2_sampled['tsne-1'] = X_medsam2_tsne[:, 0]
df_medsam2_sampled['tsne-2'] = X_medsam2_tsne[:, 1]

In [ ]:
# ---------------------------
# Combined figure: rows = color-by variable, cols = model.
# Color is the ONLY encoding channel (no fill/no-fill split by diagnosis) -
# each point is a plain filled circle so color unambiguously means one thing per row.
# ---------------------------
CAT_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEPARATOR_COLOR = "#c3c2b7"

def sort_key(v):
    try:
        return (0, float(v))
    except (TypeError, ValueError):
        return (1, str(v))

row_specs = [
    (label_col, "Diagnosis", {0: "CN", 1: "AD"}),
    (cohort_col, "Cohort", None),
    (scanner_col, "Scanner Manufacturer", None),
    (mfs_col, "Field Strength (T)", None),
]
model_specs = [
    ("SAM-Med3D", df_sammed_sampled),
    ("BrainIAC", df_brainiac_sampled),
    ("Medical-SAM2", df_medsam2_sampled),
]

fig, axes = plt.subplots(
    len(row_specs), len(model_specs),
    figsize=(6 * len(model_specs), 4.5 * len(row_specs)),
    squeeze=False,
)

for i, (col_key, row_title, value_map) in enumerate(row_specs):
    values = sorted(
        set(df_sammed_sampled[col_key].unique()) | set(df_brainiac_sampled[col_key].unique()) | set(df_medsam2_sampled[col_key].unique()),
        key=sort_key,
    )
    color_map = {v: CAT_COLORS[k % len(CAT_COLORS)] for k, v in enumerate(values)}
    display_name = (lambda v: value_map.get(v, v)) if value_map else (lambda v: v)

    legend_handles = {}
    for j, (model_name, df_sampled) in enumerate(model_specs):
        ax = axes[i, j]
        # Largest category first (bottom layer) so minority categories plot last
        # and stay visible on top instead of being buried under overlap.
        counts = df_sampled[col_key].value_counts()
        plot_order = sorted(values, key=lambda v: counts.get(v, 0), reverse=True)
        for v in plot_order:
            subset = df_sampled[df_sampled[col_key] == v]
            if subset.empty:
                continue
            sc = ax.scatter(
                subset['tsne-1'], subset['tsne-2'],
                color=color_map[v],
                s=14,
                alpha=0.7,
                edgecolors='none',
                label=str(display_name(v)),
            )
            legend_handles[str(display_name(v))] = sc
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if i == 0:
            ax.set_title(model_name, fontsize=18, pad=10)
        if j == 0:
            ax.set_ylabel(row_title, fontsize=15, labelpad=10)

    # Legend order follows the fixed `values` order (not the per-panel plot order)
    # so it stays consistent between the two model columns.
    ordered_labels = [str(display_name(v)) for v in values if str(display_name(v)) in legend_handles]
    ordered_handles = [legend_handles[l] for l in ordered_labels]
    axes[i, -1].legend(
        ordered_handles, ordered_labels,
        loc='center left',
        bbox_to_anchor=(1.03, 0.5),
        fontsize=11,
        frameon=False,
        markerscale=1.8,
        title=row_title,
        title_fontsize=12,
    )

fig.subplots_adjust(hspace=0.2, wspace=0.15, right=0.82)

# ---------------------------
# Separator lines between rows/cols (grid panels touch, so spines alone don't read as separate cells)
# ---------------------------
fig.canvas.draw()  # finalize axes positions before reading them

left_edge = min(axes[i, 0].get_position().x0 for i in range(len(row_specs)))
right_edge = max(axes[i, -1].get_position().x1 for i in range(len(row_specs)))
top_edge = max(axes[0, j].get_position().y1 for j in range(len(model_specs)))
bottom_edge = min(axes[-1, j].get_position().y0 for j in range(len(model_specs)))

# vertical separators between model columns
for j in range(len(model_specs) - 1):
    x_sep = (axes[0, j].get_position().x1 + axes[0, j + 1].get_position().x0) / 2
    fig.add_artist(plt.Line2D(
        [x_sep, x_sep], [bottom_edge, top_edge],
        transform=fig.transFigure, color=SEPARATOR_COLOR, linewidth=1,
    ))

# horizontal separators between row groups
for i in range(len(row_specs) - 1):
    y_sep = (axes[i, 0].get_position().y0 + axes[i + 1, 0].get_position().y1) / 2
    fig.add_artist(plt.Line2D(
        [left_edge, right_edge], [y_sep, y_sep],
        transform=fig.transFigure, color=SEPARATOR_COLOR, linewidth=1,
    ))

fig.savefig("../../local/figures/tsne_others_combined.png", dpi=300, bbox_inches='tight')
plt.show()